# 09 - Ensemble: Weighted Soft Voting

Equal-weight voting treats a 0.745 model the same as a 0.833 model. **Weighted** voting gives stronger (and more decorrelated) models more say. We compare three ways to set weights, all scored on the same OOF split:

1. **Equal** (the baseline from notebook 08).
2. **By CV score** - weight each model by its own OOF macro-F1 (simple, no search).
3. **Searched** - coordinate-ascent over weights to directly maximize OOF macro-F1 (a weight can go to 0, effectively dropping a member).

Then per-class threshold tuning, then `outputs/ensemble_weighted_submission.csv`.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())   # make the local helper importable
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, classification_report
import ensemble_lib as eb        # shared: tuned base models + cached OOF/test probabilities

# Load each base model's probabilities. The FIRST run computes & caches them to
# outputs/preds/*.npy (slow, a few minutes); afterwards this is instant.
NAMES = ['svm', 'catboost', 'histgbm', 'logreg', 'knn']
oof  = {n: eb.get_proba(n)[0] for n in NAMES}   # leak-free train probabilities
tst  = {n: eb.get_proba(n)[1] for n in NAMES}   # test probabilities
print('\nIndividual OOF macro-F1 (baseline to beat):')
for n in NAMES:
    print(f'  {n:9s} {eb.macro_f1(oof[n]):.4f}')

[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities
[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities

Individual OOF macro-F1 (baseline to beat):
  svm       0.8324
  catboost  0.8216
  histgbm   0.8165
  logreg    0.7453
  knn       0.7641


## Define the candidate pool
We use the three strong models; weight search is free to down-weight the redundant one. (You can add 'logreg'/'knn' to MEMBERS to let the search decide on them too.)

In [2]:
MEMBERS = ['svm', 'catboost', 'histgbm']

def wvote(weights, probs):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()                # normalize (argmax is scale-free, but keeps it interpretable)
    return sum(wi * probs[m] for wi, m in zip(weights, MEMBERS))

## Scheme 1 & 2 - equal vs CV-score weighting

In [3]:
equal_w = np.ones(len(MEMBERS))
cv_w    = np.array([eb.macro_f1(oof[m]) for m in MEMBERS])   # weight = each model's OOF score

print('equal weights   ', np.round(equal_w/equal_w.sum(), 3), '-> OOF %.4f' % eb.macro_f1(wvote(equal_w, oof)))
print('cv-score weights', np.round(cv_w/cv_w.sum(), 3),       '-> OOF %.4f' % eb.macro_f1(wvote(cv_w, oof)))

equal weights    [0.333 0.333 0.333] -> OOF 0.8330
cv-score weights [0.337 0.333 0.331] -> OOF 0.8329


## Scheme 3 - searched weights (coordinate ascent on OOF macro-F1)
We sweep each member's weight over a grid, keep improvements, and repeat a few passes. This directly optimizes the metric and can zero-out a member that doesn't help.

In [4]:
grid = np.round(np.linspace(0.0, 2.0, 21), 2)
w = np.ones(len(MEMBERS))
best = eb.macro_f1(wvote(w, oof))
for _ in range(5):
    for i in range(len(MEMBERS)):
        best_f = w[i]
        for f in grid:
            cand = w.copy(); cand[i] = f
            s = eb.macro_f1(wvote(cand, oof))
            if s > best:
                best, best_f = s, f
        w[i] = best_f
search_w = w
print('searched weights', dict(zip(MEMBERS, np.round(search_w/search_w.sum(), 3))))
print('OOF macro-F1: equal %.4f | cv %.4f | searched %.4f'
      % (eb.macro_f1(wvote(equal_w, oof)), eb.macro_f1(wvote(cv_w, oof)), best))

searched weights {'svm': np.float64(0.32), 'catboost': np.float64(0.52), 'histgbm': np.float64(0.16)}
OOF macro-F1: equal 0.8330 | cv 0.8329 | searched 0.8353


## Per-class threshold tuning on the best-weighted ensemble

In [5]:
best_oof = wvote(search_w, oof)
cw, tuned = eb.tune_class_weights(best_oof)
print('class weights:', np.round(cw, 3))
print('OOF before class-tuning: %.4f' % eb.macro_f1(best_oof))
print('OOF after  class-tuning: %.4f' % tuned)
print(classification_report(eb.y, (best_oof * cw).argmax(1), digits=3))

class weights: [0.9 1.  1.  1.1]
OOF before class-tuning: 0.8353
OOF after  class-tuning: 0.8363
              precision    recall  f1-score   support

           0      0.877     0.856     0.866      2001
           1      0.851     0.857     0.854      2442
           2      0.794     0.779     0.787      2237
           3      0.825     0.852     0.839      2320

    accuracy                          0.836      9000
   macro avg      0.837     0.836     0.836      9000
weighted avg      0.836     0.836     0.836      9000



## Build the submission

In [6]:
os.makedirs('../outputs', exist_ok=True)
final = (wvote(search_w, tst) * cw).argmax(1).astype(int)
sub = pd.DataFrame({'id': eb.test['id'], 'sleep_stage': final})
sub.to_csv('../outputs/ensemble_weighted_submission.csv', index=False)
print('wrote ../outputs/ensemble_weighted_submission.csv', sub.shape)
print('predicted class distribution:', dict(sub.sleep_stage.value_counts().sort_index()))
sub.head()

wrote ../outputs/ensemble_weighted_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1100), 1: np.int64(1295), 2: np.int64(1290), 3: np.int64(1315)}


,id,sleep_stage
0,9000,0
1,9001,3
2,9002,1
3,9003,2
4,9004,3


## Takeaways
- Searched weights >= cv-score weights >= equal weights on OOF (by construction the search can only match-or-beat equal).
- Watch for overfitting: if searched weights barely beat equal on OOF, prefer equal/cv weights for a more robust private-leaderboard result.
- Next: **10_stacking** lets a meta-model learn nonlinear combinations and gives the weak models a chance to contribute.